# 02b — Feature engineering: `destination_port` range buckets

Folds the measured win from `exp_port_buckets.ipynb` into the pipeline. **Non-destructive**: loads `02`'s persisted split, re-encodes the port, and writes a new *featured* split to `data/processed/featured/` for modeling. `02`'s raw split stays untouched (`03`/`04` still use it).

**Why loading the already-split data is safe:** bucketing uses **fixed IANA thresholds** (0–1023 / 1024–49151 / 49152–65535), so it's a **row-local** op (Section 7 #10) — a row's bucket doesn't depend on any other row. Applying it after the split is identical to applying it before → no leakage.

In [ ]:
import pandas as pd
from pathlib import Path

proc = Path('../data/processed')
X_train = pd.read_parquet(proc / 'X_train.parquet')
X_test  = pd.read_parquet(proc / 'X_test.parquet')
y_train = pd.read_parquet(proc / 'y_train.parquet')
y_test  = pd.read_parquet(proc / 'y_test.parquet')
print('loaded raw split:', X_train.shape, '|', X_test.shape)

## 1. Bucket the port (IANA ranges, one-hot)

Replace `destination_port` with three 0/1 columns: `port_well_known` (0–1023), `port_registered` (1024–49151), `port_ephemeral` (49152–65535).

In [ ]:
bins, labels = [-1, 1023, 49151, 65535], ['well_known', 'registered', 'ephemeral']

def bucket_port(X):
    b = pd.cut(X['destination_port'], bins=bins, labels=labels)
    oh = pd.get_dummies(b, prefix='port').astype(int)
    return X.drop(columns=['destination_port']).join(oh)

X_train = bucket_port(X_train)
X_test  = bucket_port(X_test).reindex(columns=X_train.columns, fill_value=0)

# self-check: exactly one bucket per row (row-local, no NaN); feature count 65 -> 67
port_cols = ['port_well_known', 'port_registered', 'port_ephemeral']
one_each = bool(X_train[port_cols].sum(axis=1).eq(1).all() and X_test[port_cols].sum(axis=1).eq(1).all())
print('every row in exactly one bucket:', one_each)
print('features now:', X_train.shape[1], '(was 65)')
print('bucket counts (train):')
print(X_train[port_cols].sum().to_string())

## 2. Persist the featured split

Save to `data/processed/featured/` — the modeling input going forward (Sprint 5 SVM, and any re-run DT). Labels are unchanged; copied alongside so a modeling notebook loads everything from one place.

In [ ]:
out = proc / 'featured'
out.mkdir(parents=True, exist_ok=True)
X_train.to_parquet(out / 'X_train.parquet')
X_test.to_parquet(out / 'X_test.parquet')
y_train.to_parquet(out / 'y_train.parquet')
y_test.to_parquet(out / 'y_test.parquet')

for f in sorted(out.glob('*.parquet')):
    print(f'{f.name:16} {pd.read_parquet(f).shape}')
print('malicious frac  ->  train: %.4f   test: %.4f' % (
    pd.read_parquet(out / 'y_train.parquet')['label_binary'].mean(),
    pd.read_parquet(out / 'y_test.parquet')['label_binary'].mean()))